In [9]:
import time
import math
import requests
import pandas as pd
import numpy as np
import json
from scipy.optimize import curve_fit
from datetime import datetime, timezone, timedelta
from bot_template import BaseBot, OrderBook, OrderRequest, Side, Trade

In [10]:
LONDON_LAT, LONDON_LON = 51.5074, -0.1278
THAMES_MEASURE = "0006-level-tidal_level-i-15_min-mAOD"

In [11]:
def get_thames(limit=200):
    """Fetch recent Thames tidal readings at Westminster.

    Returns DataFrame with: time, level (mAOD).
    Use limit=400 for ~4 days of history.
    """
    resp = requests.get(
        f"https://environment.data.gov.uk/flood-monitoring/id/measures/{THAMES_MEASURE}/readings",
        params={"_sorted": "", "_limit": limit},
    )
    resp.raise_for_status()
    items = resp.json().get("items", [])
    df = pd.DataFrame(items)[["dateTime", "value"]].rename(columns={"dateTime": "time", "value": "level"})
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Europe/London")
    return df.sort_values("time").reset_index(drop=True)

In [12]:
df = get_thames()
df

,time,level
0,2026-02-27 06:15:00+00:00,-0.121
1,2026-02-27 06:30:00+00:00,0.144
2,2026-02-27 06:45:00+00:00,0.437
3,2026-02-27 07:00:00+00:00,0.756
4,2026-02-27 07:15:00+00:00,1.101
...,...,...
195,2026-03-01 07:00:00+00:00,-2.130
196,2026-03-01 07:15:00+00:00,-2.122
197,2026-03-01 07:30:00+00:00,-2.054
198,2026-03-01 07:45:00+00:00,-1.896


In [6]:
def get_weather(past_steps=96, forecast_steps=96):
    """15-min weather for London. 96 steps = 24 hours.

    Returns DataFrame with: time, temperature, wind_speed, humidity,
    precipitation, cloud_cover, visibility, apparent_temperature.
    """
    variables = "temperature_2m,apparent_temperature,relative_humidity_2m,precipitation,wind_speed_10m,cloud_cover,visibility"
    resp = requests.get("https://api.open-meteo.com/v1/forecast", params={
        "latitude": LONDON_LAT, "longitude": LONDON_LON,
        "minutely_15": variables,
        "past_minutely_15": past_steps,
        "forecast_minutely_15": forecast_steps,
        "timezone": "Europe/London",
    })
    resp.raise_for_status()
    m = resp.json()["minutely_15"]
    return pd.DataFrame({
        "time": pd.to_datetime(m["time"]).tz_localize("Europe/London"),
        "temperature": m["temperature_2m"],
        "apparent_temperature": m["apparent_temperature"],
        "humidity": m["relative_humidity_2m"],
        "precipitation": m["precipitation"],
        "wind_speed": m["wind_speed_10m"],
        "cloud_cover": m["cloud_cover"],
        "visibility": m["visibility"],
    })

def celsius_to_fahrenheit(c: float) -> float:
    """Convert Open-Meteo Celsius to Fahrenheit for settlement."""
    return (c * 9/5) + 32

In [7]:
df = get_weather(96, 96)
theos = {}

# 2. WX_SPOT: Target Sunday 12:00 PM specifically
target_time = pd.Timestamp("2026-03-01 12:00:00", tz="Europe/London")

# Find the row closest to our target settlement time
settlement_row = df.iloc[(df['time'] - target_time).abs().argsort()[:1]]

if not settlement_row.empty:
    temp_c = settlement_row['temperature'].values[0]
    humidity = settlement_row['humidity'].values[0]
    temp_f = celsius_to_fahrenheit(temp_c)
    
    # Settlement formula: temp_F * humidity_%
    theos["WX_SPOT"] = temp_f * humidity

# 3. WX_SUM: Sum of (temp_F * humidity_%) / 100 over the 24h session
# Define the 24h session window (e.g., Saturday 12pm to Sunday 12pm)
session_start = target_time - pd.Timedelta(hours=24)
session_df = df[(df['time'] >= session_start) & (df['time'] <= target_time)].copy()

if not session_df.empty:
    # Apply formula to each 15-min interval in the session
    session_df['interval_val'] = session_df.apply(
        lambda row: celsius_to_fahrenheit(row['temperature']) * row['humidity'], 
        axis=1
    )
    theos["WX_SUM"] = session_df['interval_val'].sum() / 100.0

In [17]:
session_df

,time,temperature,apparent_temperature,humidity,precipitation,wind_speed,cloud_cover,visibility,interval_val
82,2026-02-28 12:00:00+00:00,9.3,5.9,72,0.0,14.8,93,16540.0,3509.28
83,2026-02-28 12:15:00+00:00,9.4,6.0,71,0.0,14.8,93,17140.0,3473.32
84,2026-02-28 12:30:00+00:00,9.5,5.9,71,0.0,15.5,95,17740.0,3486.10
85,2026-02-28 12:45:00+00:00,9.5,5.8,70,0.0,16.2,97,18340.0,3437.00
86,2026-02-28 13:00:00+00:00,9.5,5.7,69,0.0,16.6,98,18940.0,3387.90
...,...,...,...,...,...,...,...,...,...
174,2026-03-01 11:00:00+00:00,10.4,7.5,86,0.0,16.9,100,12800.0,4361.92
175,2026-03-01 11:15:00+00:00,10.5,7.5,86,0.0,17.3,100,13520.0,4377.40
176,2026-03-01 11:30:00+00:00,10.5,7.4,86,0.0,18.0,100,14240.0,4377.40
177,2026-03-01 11:45:00+00:00,10.5,7.4,87,0.0,18.4,100,14940.0,4428.30


In [14]:
df.iloc[178]

time                    2026-03-01 12:00:00+00:00
temperature                                  10.5
apparent_temperature                          7.3
humidity                                       87
precipitation                                 0.0
wind_speed                                   18.7
cloud_cover                                   100
visibility                                15660.0
Name: 178, dtype: object

In [16]:
celsius_to_fahrenheit(10.5)*87

4428.3

In [33]:
arrivals = pd.read_csv('data/arrivals.csv')
# Assuming columns: scheduled_departure_times, revised_departure_times
departures = pd.read_csv('data/departures.csv')

# 2. Define the 24h Window (Sunday 12:00 PM back to Saturday 12:00 PM)
target_time = pd.Timestamp("2026-03-01 12:00:00", tz="Europe/London")
session_start = target_time - pd.Timedelta(hours=24)

def to_london_dt(series):
    # parse_dates doesn't always handle offsets well, so we convert manually
    dt = pd.to_datetime(series, utc=True)
    return dt.dt.tz_convert("Europe/London")

arr_times = to_london_dt(arrivals["final_arrival_time"])
dep_times = to_london_dt(departures["final_departure_time"])

# Filter for the session window
arr_session = arr_times[(arr_times >= session_start) & (arr_times <= target_time)]
dep_session = dep_times[(dep_times >= session_start) & (dep_times <= target_time)]

# 3. LHR_COUNT: Total arrivals + departures
print("LHR_COUNT: ", len(arr_session) + len(dep_session))

# 4. LHR_INDEX: Imbalance metric per 30-min interval
# Formula: abs(sum(100 * (arr - dep) / (arr + dep)))

# 1. Initialize the bins (48 intervals in 24 hours)
bins = pd.date_range(start=session_start, end=target_time, freq='30min')

# 2. Create the base DataFrame using the interval start times
# We use bins[:-1] because the last timestamp is the end of the final interval
df_metrics = pd.DataFrame({'interval_start': bins[:-1]})

# 3. Calculate Arrivals and Departures per bin
# We use a lambda to count how many timestamps fall between start and end
df_metrics['n_arr'] = df_metrics['interval_start'].apply(
    lambda s: len(arr_session[(arr_session >= s) & (arr_session < s + pd.Timedelta(minutes=30))])
)

df_metrics['n_dep'] = df_metrics['interval_start'].apply(
    lambda s: len(dep_session[(dep_session >= s) & (dep_session < s + pd.Timedelta(minutes=30))])
)

# 4. Calculate Interval Metric
# Formula: 100 * (arr - dep) / (arr + dep)
def calc_imbalance(row):
    total = row['n_arr'] + row['n_dep']
    if total > 0:
        return 100 * (row['n_arr'] - row['n_dep']) / total
    return 0

df_metrics['interval_metric'] = df_metrics.apply(calc_imbalance, axis=1)

# 5. Rolling Sum (The 'building' total toward settlement)
df_metrics['rolling_sum'] = df_metrics['interval_metric'].cumsum()
df_metrics['tot num'] = (df_metrics['n_arr'] + df_metrics['n_dep']).cumsum()

# 6. Final Theo (Absolute value of the total sum)
# self.theos["LHR_INDEX"] = abs(df_metrics['rolling_sum'].iloc[-1])

df_metrics

LHR_COUNT:  1190


,interval_start,n_arr,n_dep,interval_metric,rolling_sum,tot num
0,2026-02-28 12:00:00+00:00,19,22,-7.317073,-7.317073,41
1,2026-02-28 12:30:00+00:00,22,22,0.000000,-7.317073,85
2,2026-02-28 13:00:00+00:00,18,20,-5.263158,-12.580231,123
3,2026-02-28 13:30:00+00:00,17,21,-10.526316,-23.106547,161
4,2026-02-28 14:00:00+00:00,20,21,-2.439024,-25.545571,202
5,2026-02-28 14:30:00+00:00,19,21,-5.000000,-30.545571,242
6,2026-02-28 15:00:00+00:00,18,19,-2.702703,-33.248274,279
7,2026-02-28 15:30:00+00:00,26,18,18.181818,-15.066456,323
8,2026-02-28 16:00:00+00:00,20,21,-2.439024,-17.505480,364
9,2026-02-28 16:30:00+00:00,14,22,-22.222222,-39.727702,400
